# SBench Harness Metrics Comparison

This notebook compares `bdi`, `codex`, and `opencode` using the metrics recorded in `results.sqlite`. It focuses on observable run metrics: completion, deliverables, elapsed time, token usage, cache usage, and reported call counts.


## 1. Configuration And Data Load

The database is opened read-only. Run this notebook from the repository root or from `notebooks/`; the root is discovered automatically.


In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, PercentFormatter

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    display = print

HARNESS_ORDER = ['bdi', 'codex', 'opencode']
HARNESS_COLORS = {
    'bdi': '#4C78A8',
    'codex': '#F58518',
    'opencode': '#54A24B',
}

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')


In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'sbench').is_dir() and (path / 'results.sqlite').exists():
            return path
    raise FileNotFoundError('Could not find repo root containing sbench/ and results.sqlite')


REPO_ROOT = find_repo_root(Path.cwd())
DATABASE_PATH = REPO_ROOT / 'results.sqlite'
DATABASE_URI = DATABASE_PATH.resolve().as_uri() + '?mode=ro'

query = """
SELECT
    execution_id,
    run_id,
    task_id,
    track,
    harness,
    model,
    archive_run,
    status,
    timed_out,
    timeout_seconds,
    elapsed_seconds,
    elapsed_bucket,
    archived_file_count,
    deliverables_present,
    token_usage_available,
    token_usage_source,
    token_total,
    token_input,
    token_output,
    token_cached_input,
    token_cache_read,
    token_cache_write,
    token_reasoning,
    llm_calls,
    tool_calls
FROM executions
WHERE harness IN ('bdi', 'codex', 'opencode')
ORDER BY task_id, harness, archive_run
"""

with sqlite3.connect(DATABASE_URI, uri=True) as connection:
    runs = pd.read_sql_query(query, connection)

missing_harnesses = set(HARNESS_ORDER) - set(runs['harness'])
if missing_harnesses:
    raise ValueError(f'Missing harness data: {sorted(missing_harnesses)}')

numeric_columns = [
    'timed_out',
    'timeout_seconds',
    'elapsed_seconds',
    'archived_file_count',
    'deliverables_present',
    'token_usage_available',
    'token_total',
    'token_input',
    'token_output',
    'token_cached_input',
    'token_cache_read',
    'token_cache_write',
    'token_reasoning',
    'llm_calls',
    'tool_calls',
]
for column in numeric_columns:
    runs[column] = pd.to_numeric(runs[column], errors='coerce')

runs['harness'] = pd.Categorical(runs['harness'], categories=HARNESS_ORDER, ordered=True)
runs['success'] = runs['status'].eq('success')
runs['timed_out'] = runs['timed_out'].fillna(0).astype(bool)
runs['deliverables_present'] = runs['deliverables_present'].fillna(0).astype(bool)
runs['task_label'] = runs['task_id'].str.replace('_', ' ').str.title()
runs['track_label'] = runs['track'].str.replace('_', ' ').str.title()
runs['reported_call_count'] = runs['llm_calls'].combine_first(runs['tool_calls'])

cache_columns = ['token_cached_input', 'token_cache_read', 'token_cache_write']
runs['cache_tokens_observed'] = runs[cache_columns].fillna(0).max(axis=1)
runs['cache_input_ratio'] = np.where(
    runs['token_input'] > 0,
    runs['cache_tokens_observed'] / runs['token_input'],
    np.nan,
)

dataset_summary = {
    'database': str(DATABASE_PATH),
    'runs': len(runs),
    'tasks': runs['task_id'].nunique(),
    'tracks': runs['track'].nunique(),
    'harness_counts': runs['harness'].value_counts(sort=False).to_dict(),
    'models': sorted(runs['model'].dropna().unique()),
}
dataset_summary


## 2. Summary By Harness

The table below aggregates the metrics that are comparable across runs. Missing call metrics mean that the harness did not report that field in `results.sqlite`.


In [ ]:
def percent_mean(series: pd.Series) -> float:
    return 100.0 * series.mean()


harness_summary = runs.groupby('harness', observed=False).agg(
    runs=('execution_id', 'count'),
    tasks=('task_id', 'nunique'),
    success_rate=('success', percent_mean),
    deliverables_rate=('deliverables_present', percent_mean),
    timeout_rate=('timed_out', percent_mean),
    avg_elapsed_seconds=('elapsed_seconds', 'mean'),
    median_elapsed_seconds=('elapsed_seconds', 'median'),
    avg_token_total=('token_total', 'mean'),
    median_token_total=('token_total', 'median'),
    avg_cache_input_pct=('cache_input_ratio', percent_mean),
    avg_llm_calls=('llm_calls', 'mean'),
    avg_tool_calls=('tool_calls', 'mean'),
    avg_reported_call_count=('reported_call_count', 'mean'),
).reindex(HARNESS_ORDER)

display(harness_summary.round(2))


## 3. Completion And Deliverables

These bars show whether each harness finished successfully, produced the requested deliverables, and avoided timeouts. In the current data all runs succeeded and produced deliverables, so the later charts focus on efficiency differences.


In [ ]:
outcome_rates = harness_summary[['success_rate', 'deliverables_rate']].rename(
    columns={
        'success_rate': 'success',
        'deliverables_rate': 'deliverables present',
    }
)

fig, ax = plt.subplots(figsize=(8, 4))
outcome_rates.plot(
    kind='bar',
    ax=ax,
    color=['#4C78A8', '#72B7B2'],
    width=0.72,
)
ax.set_title('Completion and deliverable rates by harness')
ax.set_ylabel('Rate')
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel('')
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', padding=3, fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

timeout_summary = harness_summary[['timeout_rate']].rename(columns={'timeout_rate': 'timeout rate'})
display(timeout_summary.round(2))


## 4. Elapsed Time

The first chart shows mean elapsed seconds by task and harness. The second chart shows the run-level distribution for each harness.


In [ ]:
task_order = runs.groupby('task_id')['elapsed_seconds'].mean().sort_values().index
task_label_order = [
    runs.loc[runs['task_id'].eq(task_id), 'task_label'].iloc[0]
    for task_id in task_order
]

elapsed_by_task = runs.pivot_table(
    index='task_label',
    columns='harness',
    values='elapsed_seconds',
    aggfunc='mean',
    observed=False,
).loc[task_label_order, HARNESS_ORDER]

fig, ax = plt.subplots(figsize=(11, 5))
elapsed_by_task.plot(
    kind='bar',
    ax=ax,
    color=[HARNESS_COLORS[harness] for harness in HARNESS_ORDER],
    width=0.78,
)
ax.set_title('Mean elapsed time by task')
ax.set_ylabel('Seconds')
ax.set_xlabel('')
ax.legend(title='Harness')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
elapsed_data = [
    runs.loc[runs['harness'].eq(harness), 'elapsed_seconds'].dropna()
    for harness in HARNESS_ORDER
]
box = ax.boxplot(elapsed_data, tick_labels=HARNESS_ORDER, patch_artist=True, showmeans=True)
for patch, harness in zip(box['boxes'], HARNESS_ORDER):
    patch.set_facecolor(HARNESS_COLORS[harness])
    patch.set_alpha(0.65)
ax.set_title('Elapsed time distribution by harness')
ax.set_ylabel('Seconds')
plt.tight_layout()
plt.show()


## 5. Token Usage

Token totals are compared both by task and as a run-level distribution. The y-axis is formatted in thousands of tokens.


In [ ]:
def thousands_formatter(value: float, _: int) -> str:
    return f'{value / 1000:.0f}k'


token_by_task = runs.pivot_table(
    index='task_label',
    columns='harness',
    values='token_total',
    aggfunc='mean',
    observed=False,
).loc[task_label_order, HARNESS_ORDER]

fig, ax = plt.subplots(figsize=(11, 5))
token_by_task.plot(
    kind='bar',
    ax=ax,
    color=[HARNESS_COLORS[harness] for harness in HARNESS_ORDER],
    width=0.78,
)
ax.set_title('Mean total tokens by task')
ax.set_ylabel('Tokens')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
ax.legend(title='Harness')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
token_data = [
    runs.loc[runs['harness'].eq(harness), 'token_total'].dropna()
    for harness in HARNESS_ORDER
]
box = ax.boxplot(token_data, tick_labels=HARNESS_ORDER, patch_artist=True, showmeans=True)
for patch, harness in zip(box['boxes'], HARNESS_ORDER):
    patch.set_facecolor(HARNESS_COLORS[harness])
    patch.set_alpha(0.65)
ax.set_title('Total token distribution by harness')
ax.set_ylabel('Tokens')
ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
plt.tight_layout()
plt.show()


## 6. Cache And Call Instrumentation

Cache metrics are not emitted in exactly the same field by every harness, so `cache_tokens_observed` uses the largest reported cached-token field per run. Call metrics are plotted as separate `llm_calls` and `tool_calls` fields; blanks mean the field was not reported for that harness.


In [ ]:
cache_summary = runs.groupby('harness', observed=False).agg(
    avg_cache_tokens=('cache_tokens_observed', 'mean'),
    avg_cache_input_pct=('cache_input_ratio', percent_mean),
).reindex(HARNESS_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cache_summary['avg_cache_tokens'].plot(
    kind='bar',
    ax=axes[0],
    color=[HARNESS_COLORS[harness] for harness in HARNESS_ORDER],
)
axes[0].set_title('Mean observed cached tokens')
axes[0].set_ylabel('Tokens')
axes[0].yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

cache_summary['avg_cache_input_pct'].plot(
    kind='bar',
    ax=axes[1],
    color=[HARNESS_COLORS[harness] for harness in HARNESS_ORDER],
)
axes[1].set_title('Mean cached share of input tokens')
axes[1].set_ylabel('Input tokens cached')
axes[1].yaxis.set_major_formatter(PercentFormatter(100))
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

display(cache_summary.round(2))


In [ ]:
call_summary = runs.groupby('harness', observed=False).agg(
    avg_llm_calls=('llm_calls', 'mean'),
    avg_tool_calls=('tool_calls', 'mean'),
    avg_reported_call_count=('reported_call_count', 'mean'),
).reindex(HARNESS_ORDER)

fig, ax = plt.subplots(figsize=(8, 4.5))
call_summary[['avg_llm_calls', 'avg_tool_calls']].rename(
    columns={
        'avg_llm_calls': 'LLM calls',
        'avg_tool_calls': 'tool calls',
    }
).plot(
    kind='bar',
    ax=ax,
    color=['#B279A2', '#9D755D'],
    width=0.72,
)
ax.set_title('Mean reported calls by harness')
ax.set_ylabel('Calls per run')
ax.set_xlabel('')
ax.legend(loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(call_summary.round(2))


## 7. Time vs Tokens

Each point below is one execution. This view is useful for quickly spotting whether a harness spends more tokens, more wall time, or both.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for harness in HARNESS_ORDER:
    subset = runs.loc[runs['harness'].eq(harness)]
    ax.scatter(
        subset['elapsed_seconds'],
        subset['token_total'],
        s=80,
        alpha=0.78,
        color=HARNESS_COLORS[harness],
        label=harness,
        edgecolor='white',
        linewidth=0.8,
    )

ax.set_title('Elapsed time vs total tokens by execution')
ax.set_xlabel('Elapsed seconds')
ax.set_ylabel('Total tokens')
ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
ax.legend(title='Harness')
plt.tight_layout()
plt.show()


## 8. Suggested Slide Takeaways

Use the table and charts above to choose the exact claims for a slide. With the current `results.sqlite`, the high-level pattern is:

- All three harnesses completed the benchmark runs and produced deliverables.
- `codex` is fastest on average in this dataset, but uses substantially more total tokens.
- `opencode` uses the fewest total tokens on average, with more reported LLM calls.
- `bdi` sits between `codex` and `opencode` on tokens, but is slower on average and reports tool-call style activity instead of LLM-call counts.
- Treat call-count comparisons carefully because each harness exposes different instrumentation fields.
